In [1]:
from ndpi.visualization.tables import fix_gt_table

from ech_discovery.notebook import *

# Load data

In [2]:
df_versions = pl.read_csv(
    config.INTERIM_DATA_DIR / "webserver-versions.tsv",
    separator="\t",
    infer_schema_length=None,
)

In [3]:
df = (
    pl.read_csv(config.INTERIM_DATA_DIR / "tls-attacker-results.tsv", separator="\t")
    .join(df_versions, on=[C.server, C.version], how="left", validate="m:1")
    .with_columns(
        pl.col(C.server).replace("apache", Strings.apache_httpd),
    )
)
assert (
    df[C.released].null_count() == 0
), f"unmatched versions: {df.filter(pl.col(C.released).is_null()).select(C.server, C.version).rows()}"
with pl.Config(tbl_rows=50):
    display(df)

server,version,variant,TLS-grease,niere-unknown-extension-with-value,parsable-ech-extension,unparsable-ech-as-raw-extension,verdict,released,source
str,str,str,str,str,str,str,str,str,str
"""Apache httpd""","""2.4.67""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2026-05-04""","""https://archive.apache.org/dis…"
"""Apache httpd""","""trunk""","""ech""","""-""","""-""","""unk-ee""","""decode_error""","""ECH-aware (rejects 0xFE0D with…","""2026-03-23""","""defo-project/apache-httpd trun…"
"""Apache httpd""","""trunk""","""ech-disabled""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2026-03-23""","""defo-project/apache-httpd trun…"
"""nginx""","""1.21.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2021-05-25""","""https://nginx.org/en/CHANGES (…"
"""nginx""","""1.22.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2022-05-24""","""https://nginx.org/en/CHANGES (…"
"""nginx""","""1.24.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2023-04-11""","""https://nginx.org/en/CHANGES (…"
"""nginx""","""1.26.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2024-04-23""","""https://nginx.org/en/CHANGES (…"
"""nginx""","""1.27.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2024-05-29""","""https://nginx.org/en/CHANGES (…"
"""nginx""","""1.28.0""","""base""","""-""","""-""","""-""","""-""","""Not ECH-aware""","""2025-04-23""","""https://nginx.org/en/CHANGES (…"


# Table output 

In [4]:
df_gt = (
    df.sort(C.server, C.version, C.variant)
    .with_columns(
        (
            pl.when(pl.col(C.variant).str.contains("ech"))
            .then(pl.lit(Strings.openssl3_ech))
            .when(pl.col(C.variant).str.contains("base"))
            .then(pl.lit(Strings.openssl3))
            .when(pl.col(C.variant).str.contains("boring"))
            .then(pl.lit(Strings.boringssl))
        ).alias(Strings.tls_library),
        (
            pl.when(pl.col(C.variant).str.contains("disabled|base"))
            .then(pl.lit("-"))
            .when(pl.col(C.variant).is_in(["ech", "boringssl"]))
            .then(pl.lit("x"))
            .alias(Strings.ech_configured)
        ),
        pl.when(pl.col(C.unparsable_ech_extension) != "-")
        .then(pl.lit("x"))
        .fill_null("-")
        .alias(Strings.niere_et_al),
        pl.when(pl.col(C.parsable_ech_extension) != "-")
        .then(pl.lit("x"))
        .fill_null("-")
        .alias(Strings.this_work),
        (
            pl.when(
                pl.col(C.variant).str.contains("ech")
                | pl.col(C.variant).str.contains("boringssl")
            ).then(pl.lit("x"))
        )
        .fill_null("-")
        .alias(Strings.code_support),
    )
    .filter(~(pl.col(C.variant) == "boringssl"))
    .select(
        Strings.tls_library,
        C.server,
        C.version,
        C.released,
        Strings.code_support,
        Strings.ech_configured,
        C.niere_unknown_extension,
        C.unparsable_ech_extension,
        C.parsable_ech_extension,
        Strings.niere_et_al,
        Strings.this_work,
        C.verdict,
    )
    .rename(
        {
            C.niere_unknown_extension: Strings.niere_unknown_extension,
            C.unparsable_ech_extension: Strings.unparsable_ech_extension,
            C.parsable_ech_extension: Strings.parsable_ech_extension,
            C.server: Strings.webserver,
            C.version: Strings.version,
            C.verdict: Strings.result,
            C.released: Strings.released,
        }
    )
    .sort(Strings.tls_library, Strings.webserver, Strings.version, Strings.code_support)
)
df_gt

TLS library,Webserver,Version,Released,Code Support,ECH Configured,Unparsacble ECH configuration in TLS grease extension (cf. Niere et al.),Unparsable ECH extension (cf. Niere et al.),Parsable ECH extension (this work),Niere et al.,This Work,Result
str,str,str,str,str,str,str,str,str,str,str,str
"""BoringSSL""","""nginx""","""1.29.4""","""2025-12-09""","""x""","""-""","""-""","""decode_error""","""-""","""x""","""-""","""ECH-aware (rejects 0xFE0D with…"
"""BoringSSL""","""nginx""","""1.30.0""","""2026-04-14""","""x""","""-""","""-""","""decode_error""","""-""","""x""","""-""","""ECH-aware (rejects 0xFE0D with…"
"""BoringSSL""","""nginx""","""1.31.0""","""2026-05-13""","""x""","""-""","""-""","""decode_error""","""-""","""x""","""-""","""ECH-aware (rejects 0xFE0D with…"
…,…,…,…,…,…,…,…,…,…,…,…
"""OpenSSL3 with ECH (defo fork)""","""nginx""","""1.31.0""","""2026-05-13""","""x""","""x""","""-""","""decode_error""","""unk-ee""","""x""","""x""","""ECH-aware (rejects 0xFE0D with…"
"""OpenSSL3 with ECH (defo fork)""","""nginx""","""1.31.0""","""2026-05-13""","""x""","""-""","""-""","""-""","""-""","""-""","""-""","""Not ECH-aware"""


In [5]:
def add_group_midrules(latex: str, group_values: list) -> str:
    lines = latex.split("\n")
    bottomrule_idx = next(
        i for i, l in enumerate(lines) if l.strip().startswith(r"\bottomrule")
    )
    body_start = max(i for i in range(bottomrule_idx) if r"\midrule" in lines[i])
    body_lines = lines[body_start + 1 : bottomrule_idx]

    assert len(body_lines) == len(
        group_values
    ), f"Row count mismatch: {len(body_lines)} body lines vs {len(group_values)} group values"

    new_body = []
    for i, (line, val) in enumerate(zip(body_lines, group_values)):
        new_body.append(line)
        is_last_row = i == len(group_values) - 1
        if not is_last_row and val != group_values[i + 1]:
            new_body.append(r"\midrule")

    lines[body_start + 1 : bottomrule_idx] = new_body
    return "\n".join(lines)

In [6]:
cols = df_gt.columns
title = """
Apache httpd and nginx tested with TLS Attacker reproducing the evaluation of Niere et al.
TLS Attacker correctly determines code support for BoringSSL.
With OpenSSL, ECH is not detected by Niere et al. due to OpenSSL ignoring errors in used ECH extensions.
Our work does not detect code support, but whether ECH is configured or not.
We do not test BoringSSL builds with ECH configured, because the webservers lack the respective bindings for BoringSSL.
"""

ECH_DETECTED_SYMBOL = {"x": Strings.latex_cmark.value, "-": Strings.latex_xmark.value}
ECH_DETECTED_COLS = [
    Strings.code_support,
    Strings.ech_configured,
    Strings.niere_et_al,
    Strings.this_work,
]

gt = (
    GT(df_gt.drop(cols[6:9] + [Strings.result]))
    .tab_spanner(label=Strings.server_configuration, columns=cols[:6])
    .tab_spanner(label=Strings.ech_detected, columns=cols[9:11])
    .tab_header(title=title)
    .fmt(fns=lambda x: ECH_DETECTED_SYMBOL[x], columns=ECH_DETECTED_COLS)
    .fmt(
        fns=lambda x: x.replace("(defo fork)", r"\cite{d-o-23}"),
        columns=Strings.tls_library,
    )
    .fmt(
        fns=lambda x: x.replace("trunk", r"trunk \cite{d-a-23}"),
        columns=Strings.version,
    )
    .cols_align(align="center", columns=ECH_DETECTED_COLS)
)
display(gt)
print(
    add_group_midrules(
        fix_gt_table(gt.as_latex(), "tab:tls-attacker-nginx-httpd"),
        df_gt[Strings.tls_library].to_list(),
    )
)

GT(_tbl_data=shape: (21, 8)
┌───────────────┬───────────┬─────────┬────────────┬─────────┬──────────────┬──────────┬───────────┐
│ TLS library   ┆ Webserver ┆ Version ┆ Released   ┆ Code    ┆ ECH          ┆ Niere et ┆ This Work │
│ ---           ┆ ---       ┆ ---     ┆ ---        ┆ Support ┆ Configured   ┆ al.      ┆ ---       │
│ str           ┆ str       ┆ str     ┆ str        ┆ ---     ┆ ---          ┆ ---      ┆ str       │
│               ┆           ┆         ┆            ┆ str     ┆ str          ┆ str      ┆           │
╞═══════════════╪═══════════╪═════════╪════════════╪═════════╪══════════════╪══════════╪═══════════╡
│ BoringSSL     ┆ nginx     ┆ 1.29.4  ┆ 2025-12-09 ┆ x       ┆ -            ┆ x        ┆ -         │
│ BoringSSL     ┆ nginx     ┆ 1.30.0  ┆ 2026-04-14 ┆ x       ┆ -            ┆ x        ┆ -         │
│ BoringSSL     ┆ nginx     ┆ 1.31.0  ┆ 2026-05-13 ┆ x       ┆ -            ┆ x        ┆ -         │
│ …             ┆ …         ┆ …       ┆ …          ┆ …       ┆ …            ┆ …        ┆ …         │
│ OpenSSL3 with ┆ nginx     ┆ 1.31.0  ┆ 2026-05-13 ┆ x       ┆ x            ┆ x        ┆ x         │
│ ECH (defo     ┆           ┆         ┆            ┆         ┆              ┆          ┆           │
│ fork)         ┆           ┆         ┆            ┆         ┆              ┆          ┆           │
│ OpenSSL3 with ┆ nginx     ┆ 1.31.0  ┆ 2026-05-13 ┆ x       ┆ -            ┆ -        ┆ -         │
│ ECH (defo     ┆           ┆         ┆            ┆         ┆              ┆          ┆           │
│ fork)         ┆           ┆         ┆            ┆         ┆              ┆          ┆           │
└───────────────┴───────────┴─────────┴────────────┴─────────┴──────────────┴──────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x76d98013fcb0>, _boxhead=Boxhead([ColInfo(var='TLS library', type=<ColInfoTypeEnum.default: 1>, column_label='TLS library', column_align='left', column_width=None), ColInfo(var='Webserver', type=<ColInfoTypeEnum.default: 1>, column_label='Webserver', column_align='left', column_width=None), ColInfo(var='Version', type=<ColInfoTypeEnum.default: 1>, column_label='Version', column_align='left', column_width=None), ColInfo(var='Released', type=<ColInfoTypeEnum.default: 1>, column_label='Released', column_align='left', column_width=None), ColInfo(var='Code Support', type=<ColInfoTypeEnum.default: 1>, column_label='Code Support', column_align='center', column_width=None), ColInfo(var='ECH Configured', type=<ColInfoTypeEnum.default: 1>, column_label='ECH Configured', column_align='center', column_width=None), ColInfo(var='Niere et al.', type=<ColInfoTypeEnum.default: 1>, column_label='Niere et al.', column_align='center', column_width=None), ColInfo(var='This Work', type=<ColInfoTypeEnum.default: 1>, column_label='This Work', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x76d98013f8c0>, _spanners=Spanners([SpannerInfo(spanner_id=<Strings.server_configuration: 'Server Configuration'>, spanner_level=0, spanner_label='Server Configuration', spanner_units=None, spanner_pattern=None, vars=['TLS library', 'Webserver', 'Version', 'Released', 'Code Support', 'ECH Configured'], built=None), SpannerInfo(spanner_id=<Strings.ech_detected: 'ECH Detected'>, spanner_level=0, spanner_label='ECH Detected', spanner_units=None, spanner_pattern=None, vars=['Niere et al.', 'This Work'], built=None)]), _heading=Heading(title='\nApache httpd and nginx tested with TLS Attacker reproducing the evalation of Niere et al.\nTLS Attacker correctly determines code support for BoringSSL.\nWith OpenSSL ECH is not detected by Niere et al. due to OpenSSL ignoring errors in used ECH extensions.\nOur work does not detect code support, but whether ECH is configured or not.\nWe do not test BoringSSL builds with ECH configured, because the webservers lack the respective bindings for BoringSSL.\n', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._

\begin{table*}[!t]
\caption{Apache httpd and nginx tested with TLS Attacker reproducing the evalation of Niere et al.
TLS Attacker correctly determines code support for BoringSSL.
With OpenSSL ECH is not detected by Niere et al. due to OpenSSL ignoring errors in used ECH extensions.
Our work does not detect code support, but whether ECH is configured or not.
We do not test BoringSSL builds with ECH configured, because the webservers lack the respective bindings for BoringSSL.
}
\label{tab:tls-attacker-nginx-httpd} 
\begin{tabular}{llllcccc}
\toprule
\multicolumn{6}{c}{Server Configuration} & \multicolumn{2}{c}{ECH Detected} \\ 
\cmidrule(lr){1-6} \cmidrule(lr){7-8}
TLS library & Webserver & Version & Released & Code Support & ECH Configured & Niere et al. & This Work \\ 
\midrule\addlinespace[2.5pt]
BoringSSL & nginx & 1.29.4 & 2025-12-09 & \cmark & \xmark & \cmark & \xmark \\
BoringSSL & nginx & 1.30.0 & 2026-04-14 & \cmark & \xmark & \cmark & \xmark \\
BoringSSL & nginx & 1.31.0 & 20